# Practical 2: Regression Analysis - Uber fare prediction

Pre-process the Uber fares dataset, engineer date/time and trip-distance features, check outliers and correlations, then compare Linear, Ridge and Lasso regression using R2 and RMSE.

**Dataset:** `uber.csv` (Kaggle "Uber Fares Dataset"). On Colab keep it in `MyDrive/Data-Science`; on local Jupyter keep it next to this notebook.

In [ ]:
# Where the dataset lives:
#  - Google Colab: mounts Google Drive and reads from MyDrive/Data-Science
#  - Local Jupyter: put the CSV in the same folder as this notebook
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/Data-Science'
except ImportError:
    DATA_DIR = '.'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "uber.csv"))
df.head()

In [ ]:
# Display basic info about the DataFrame to check data types and non-null counts
df.info()

In [ ]:
# Check for missing values
print("\nMissing values before handling:\n", df.isnull().sum())

In [ ]:
# Handle missing values: drop rows with any missing values
df.dropna(inplace=True)
print("\nMissing values after handling:\n", df.isnull().sum())
print("Shape after dropping nulls:", df.shape)

In [ ]:
# Convert 'pickup_datetime' to datetime objects
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Extract time-based features
df['year'] = df['pickup_datetime'].dt.year
df['month'] = df['pickup_datetime'].dt.month
df['day'] = df['pickup_datetime'].dt.day
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek # Monday=0, Sunday=6

df.head()

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in kilometers

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

# Calculate trip distance using Haversine formula
df['distance_km'] = df.apply(lambda row: haversine(
    row['pickup_latitude'], row['pickup_longitude'],
    row['dropoff_latitude'], row['dropoff_longitude']
), axis=1)

df.head()

In [ ]:
# Drop original datetime and unnecessary columns
df.drop(['Unnamed: 0', 'key', 'pickup_datetime'], axis=1, inplace=True)

df.head()

## Identify outliers

In [ ]:
# Describe fare_amount to see its distribution and potential outliers
print("\nDescriptive statistics for 'fare_amount':\n", df['fare_amount'].describe())

# Describe distance_km
print("\nDescriptive statistics for 'distance_km':\n", df['distance_km'].describe())

# Describe passenger_count
print("\nDescriptive statistics for 'passenger_count':\n", df['passenger_count'].describe())

# Visualize distributions with box plots
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.boxplot(y=df['fare_amount'])
plt.title('Box Plot of Fare Amount')

plt.subplot(1, 3, 2)
sns.boxplot(y=df['distance_km'])
plt.title('Box Plot of Distance (km)')

plt.subplot(1, 3, 3)
sns.boxplot(y=df['passenger_count'])
plt.title('Box Plot of Passenger Count')

plt.tight_layout()
plt.show()

## Check the correlation

In [ ]:
# Calculate the correlation matrix
correlation_matrix = df.corr()

# Display the correlation of all features with 'fare_amount'
print("\nCorrelation with 'fare_amount':\n", correlation_matrix['fare_amount'].sort_values(ascending=False))

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Features', fontsize=16)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Define features (X) and target (y)
# We'll drop latitude and longitude columns as distance_km already captures their essence
X = df.drop(['fare_amount', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude'], axis=1)
y = df['fare_amount']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of training features:", X_train.shape)
print("Shape of testing features:", X_test.shape)
print("Shape of training target:", y_train.shape)
print("Shape of testing target:", y_test.shape)

### 1. Linear Regression Model

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

# Evaluate Linear Regression
r2_linear = r2_score(y_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))

print("Linear Regression - R2 Score:", r2_linear)
print("Linear Regression - RMSE:", rmse_linear)

### 2. Ridge Regression Model

In [ ]:
ridge_model = Ridge(random_state=42)
ridge_model.fit(X_train, y_train)

y_pred_ridge = ridge_model.predict(X_test)

# Evaluate Ridge Regression
r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print("Ridge Regression - R2 Score:", r2_ridge)
print("Ridge Regression - RMSE:", rmse_ridge)

### 3. Lasso Regression Model

In [ ]:
lasso_model = Lasso(random_state=42)
lasso_model.fit(X_train, y_train)

y_pred_lasso = lasso_model.predict(X_test)

# Evaluate Lasso Regression
r2_lasso = r2_score(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

print("Lasso Regression - R2 Score:", r2_lasso)
print("Lasso Regression - RMSE:", rmse_lasso)

## Compare Model Scores

Let's put all the evaluation metrics together for an easy comparison.

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression'],
    'R2 Score': [r2_linear, r2_ridge, r2_lasso],
    'RMSE': [rmse_linear, rmse_ridge, rmse_lasso]
})

display(results.sort_values(by='R2 Score', ascending=False))